# Unified ML Pipeline: Data Fetching, Preprocessing, ML Analysis, and MySQL Upsert

This notebook demonstrates the complete end-to-end pipeline for company financial analysis, including:
- Data fetching from API
- Data preprocessing and feature engineering
- ML-based analysis extraction
- Upserting results into MySQL

All steps use production-ready code and integrate a real-time logger for status updates.

In [ ]:
# Import Required Libraries
import os
import json
import pandas as pd
import requests
from time import sleep
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from datetime import datetime
from tqdm import tqdm
import logging
from colorama import init, Fore, Style

# Import the realtime logger (assumes realtime_logger.py is in the same directory or in PYTHONPATH)
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), 'production')))
from realtime_logger import log_success, log_error, log_info

## 1. Data Fetching

Fetch company financial data from the API using credentials and company IDs. Results are saved as JSON files in the data directory.

In [ ]:
# Load environment variables and set up config
load_dotenv(dotenv_path=os.path.join('..', '.env'))

API_KEY = os.getenv('API_KEY')
BASE_URL = os.getenv('BASE_URL')
COMPANY_LIST_PATH = os.path.join('..', 'company_id.xlsx')
OUTPUT_DIR = os.path.join('..', 'data')
LOG_FILE = 'data_fetching.log'
REQUEST_TIMEOUT = 10
RETRY_ATTEMPTS = 3
RETRY_DELAY = 5

In [ ]:
# Data fetching functions
def load_company_ids(file_path):
    try:
        df = pd.read_excel(file_path)
        if 'company_id' not in df.columns:
            log_error("'company_id' column not found in the Excel file.")
            return []
        return df['company_id'].dropna().unique().tolist()
    except FileNotFoundError:
        log_error(f"Error: The file at {file_path} was not found.")
        return []
    except Exception as e:
        log_error(f"An error occurred while reading the Excel file: {e}")
        return []

def fetch_financial_data(company_id):
    params = {'id': company_id, 'api_key': API_KEY}
    for attempt in range(RETRY_ATTEMPTS):
        try:
            response = requests.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            if 'No Data Found' in response.text:
                log_info(f"No data found for company ID: {company_id}. Skipping.")
                return None
            return response.json()
        except requests.exceptions.Timeout:
            log_info(f"Request for {company_id} timed out. Attempt {attempt + 1} of {RETRY_ATTEMPTS}.")
        except requests.exceptions.RequestException as e:
            log_error(f"Request for {company_id} failed: {e}. Attempt {attempt + 1} of {RETRY_ATTEMPTS}.")
        if attempt < RETRY_ATTEMPTS - 1:
            sleep(RETRY_DELAY)
    log_error(f"Failed to fetch data for {company_id} after {RETRY_ATTEMPTS} attempts.")
    return None

def save_data_to_json(data, company_id, directory):
    if not os.path.exists(directory):
        os.makedirs(directory)
    file_path = os.path.join(directory, f"{company_id}.json")
    try:
        with open(file_path, "w") as f:
            json.dump(data, f, indent=4)
        log_success(f"Successfully saved data for {company_id} to {file_path}")
    except IOError as e:
        log_error(f"Failed to write data to {file_path}: {e}")

In [ ]:
# Run data fetching process
company_ids = load_company_ids(COMPANY_LIST_PATH)
if not company_ids:
    log_error("No company IDs loaded. Exiting data fetching step.")
else:
    log_info(f"Loaded {len(company_ids)} unique company IDs.")
    for company_id in tqdm(company_ids, desc="Fetching company data"):
        log_info(f"Fetching data for company: {company_id}")
        financial_data = fetch_financial_data(company_id)
        if financial_data and 'data' in financial_data:
            data_to_save = financial_data['data']
            save_data_to_json(data_to_save, company_id, OUTPUT_DIR)
        elif financial_data:
            log_info(f"Response for {company_id} does not contain a 'data' key. Saving entire response for inspection.")
            save_data_to_json(financial_data, company_id, OUTPUT_DIR)

## 2. Data Preprocessing

Process the raw JSON files, normalize and clean the data, perform feature engineering, and store master CSVs and SQLite DB for downstream analysis.

In [ ]:
# Data Preprocessing Pipeline
import glob
import numpy as np
import re
import warnings

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    if s in ['', '-', 'None', 'null', 'nan', 'NA']:
        return np.nan
    s = s.replace(',', '')
    if re.match(r'^\(.*\)$', s):
        s = '-' + s[1:-1]
    s = s.replace('%', '')
    try:
        return float(s)
    except:
        return np.nan

def process_json_file(path):
    with open(path, 'r', encoding='utf-8') as f:
        j = json.load(f)
    cid = j.get('company', {}).get('id', os.path.splitext(os.path.basename(path))[0])
    company_meta = j.get('company', {})
    cashflow = pd.DataFrame(j.get('cash_flow', []))
    balancesheet = pd.DataFrame(j.get('balance_sheet', []))
    profitloss = pd.DataFrame(j.get('profit_and_loss', []))
    analysis = pd.DataFrame(j.get('analysis', []))
    return {
        'company_meta': company_meta,
        'cashflow': cashflow,
        'balancesheet': balancesheet,
        'profitloss': profitloss,
        'analysis': analysis
    }

def run_preprocessing(data_dir, output_dir):
    files = sorted(glob.glob(os.path.join(data_dir, '*.json')))
    all_cf, all_bs, all_pl, all_an, all_meta = [], [], [], [], []
    for fp in tqdm(files, desc="Processing JSON files"):
        res = process_json_file(fp)
        if not res['cashflow'].empty:
            all_cf.append(res['cashflow'].assign(company_id=res['company_meta'].get('id', '')))
        if not res['balancesheet'].empty:
            all_bs.append(res['balancesheet'].assign(company_id=res['company_meta'].get('id', '')))
        if not res['profitloss'].empty:
            all_pl.append(res['profitloss'].assign(company_id=res['company_meta'].get('id', '')))
        if not res['analysis'].empty:
            all_an.append(res['analysis'].assign(company_id=res['company_meta'].get('id', '')))
        all_meta.append(res['company_meta'])
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    if all_cf:
        pd.concat(all_cf).to_csv(os.path.join(output_dir, 'cashflow_master.csv'), index=False)
    if all_bs:
        pd.concat(all_bs).to_csv(os.path.join(output_dir, 'balancesheet_master.csv'), index=False)
    if all_pl:
        pd.concat(all_pl).to_csv(os.path.join(output_dir, 'profitloss_master.csv'), index=False)
    if all_an:
        pd.concat(all_an).to_csv(os.path.join(output_dir, 'analysis_master.csv'), index=False)
    if all_meta:
        pd.DataFrame(all_meta).to_csv(os.path.join(output_dir, 'companies_meta.csv'), index=False)
    log_success(f"Preprocessing complete. Master files saved to {output_dir}")

# Run preprocessing
run_preprocessing(OUTPUT_DIR, os.path.join(OUTPUT_DIR, 'compiled_output'))

## 3. ML Analysis

Extract pros, cons, and key growth metrics for each company from the preprocessed data. Save results as a JSON file for MySQL upsert.

In [ ]:
# ML Analysis Extraction
def extract_analysis(company_id, data):
    company_name = data.get("company", {}).get("company_name", company_id)
    prosandcons = data.get("prosandcons", [])
    if prosandcons:
        pros = prosandcons[0].get("pros", "")
        cons = prosandcons[0].get("cons", "")
    else:
        pros, cons = "", ""
    analysis_list = data.get("analysis", [])
    analysis_json = {}
    for period, key in [("3 Years", "3"), ("5 Years", "5"), ("10 Years", "10")]:
        for a in analysis_list:
            if period in a.get("compounded_sales_growth", ""):
                analysis_json.setdefault("compounded_sales_growth", {})[key] = a["compounded_sales_growth"]
            if period in a.get("compounded_profit_growth", ""):
                analysis_json.setdefault("compounded_profit_growth", {})[key] = a["compounded_profit_growth"]
            if period in a.get("roe", ""):
                analysis_json.setdefault("roe", {})[key] = a["roe"]
    return {
        "company_id": company_id,
        "company_name": company_name,
        "pros": pros,
        "cons": cons,
        "analysis_json": analysis_json
    }

def run_ml_analysis(data_dir, output_path):
    results = []
    for fname in os.listdir(data_dir):
        if fname.endswith(".json"):
            company_id = fname.replace(".json", "")
            with open(os.path.join(data_dir, fname), "r", encoding="utf-8") as f:
                data = json.load(f)
            analysis = extract_analysis(company_id, data)
            results.append(analysis)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    log_success(f"Wrote {len(results)} company analyses to {output_path}")

# Run ML analysis
ml_analysis_output = os.path.join(OUTPUT_DIR, 'compiled_output', 'ml_analysis_results.json')
run_ml_analysis(os.path.join(OUTPUT_DIR, 'compiled_output'), ml_analysis_output)

## 4. MySQL Upsert

Upsert the ML analysis results into the MySQL `ml` table using SQLAlchemy.

In [ ]:
# MySQL Upsert Logic
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASS = os.getenv('MYSQL_PASS', '')
MYSQL_HOST = os.getenv('MYSQL_HOST', '127.0.0.1')
MYSQL_PORT = os.getenv('MYSQL_PORT', '3306')
MYSQL_DB   = os.getenv('MYSQL_DB', 'ml')

CREATE_ML_SQL = """
CREATE TABLE IF NOT EXISTS ml (
  company_id VARCHAR(255) NOT NULL PRIMARY KEY,
  company_name VARCHAR(255),
  pros JSON,
  cons JSON,
  analysis_json JSON,
  last_updated DATETIME,
  created_at DATETIME DEFAULT CURRENT_TIMESTAMP,
  updated_at DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
"""

UPSERT_ML_SQL = """
INSERT INTO ml (company_id, company_name, pros, cons, analysis_json, last_updated)
VALUES (:company_id, :company_name, :pros, :cons, :analysis_json, :last_updated)
ON DUPLICATE KEY UPDATE
  company_name = VALUES(company_name),
  pros = VALUES(pros),
  cons = VALUES(cons),
  analysis_json = VALUES(analysis_json),
  last_updated = VALUES(last_updated);
"""

def get_engine(user, password, host, port, db):
    url = f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}?charset=utf8mb4"
    return create_engine(url, pool_size=5, max_overflow=10, pool_recycle=3600)

def prepare_record(rec):
    def smart_json(val):
        if val is None or (isinstance(val, str) and val.strip() == ""):
            return '[]'
        if isinstance(val, str):
            return json.dumps(val, ensure_ascii=False)
        return json.dumps(val, ensure_ascii=False)
    return {
        'company_id': rec['company_id'],
        'company_name': rec.get('company_name'),
        'pros': smart_json(rec.get('pros', '')),
        'cons': smart_json(rec.get('cons', '')),
        'analysis_json': json.dumps(rec.get('analysis_json', {}), ensure_ascii=False),
        'last_updated': datetime.utcnow()
    }

def upsert_batch(engine, records):
    with engine.begin() as conn:
        conn.execute(text(CREATE_ML_SQL))
        prepared = [prepare_record(r) for r in records]
        conn.execute(text(UPSERT_ML_SQL), prepared)
    log_success(f"Upserted {len(records)} records into ml.")

def load_results(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, dict) and 'results' in data:
        return data['results']
    return data

# Run MySQL upsert
engine = get_engine(MYSQL_USER, MYSQL_PASS, MYSQL_HOST, MYSQL_PORT, MYSQL_DB)
recs = load_results(ml_analysis_output)
upsert_batch(engine, recs)